# Monte Carlo LLM
- Modelling Autoregressive Next Token Prediction as a Search Task through Reinforcement Learning
- Using Llama 3.2 1B on Hugging Face

Next Steps:
1. Make an autoregressive word by word "decode" function which yields a list of "word": prob for top k words
2. Generate a graph
3. Generate a graph for top k words for top 3 choices to visualize
4. Implement and visualize temperature

In [40]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")

In [41]:
tokenized_input = tokenizer("The meaning of life",
                            return_tensors="pt").input_ids
print(f"Tokenized Embedding: {tokenized_input}")

Tokenized Embedding: tensor([[128000,    791,   7438,    315,   2324]])


In [42]:
output_logits = model.generate(tokenized_input, return_dict_in_generate=True,
    output_scores=True, max_length=50)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [43]:
print(dir(output_logits))
print(output_logits)

['__annotations__', '__class__', '__class_getitem__', '__contains__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__ior__', '__iter__', '__le__', '__len__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__or__', '__post_init__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__ror__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshook__', 'attentions', 'clear', 'copy', 'fromkeys', 'get', 'hidden_states', 'items', 'keys', 'logits', 'move_to_end', 'past_key_values', 'pop', 'popitem', 'scores', 'sequences', 'setdefault', 'to_tuple', 'update', 'values']
GenerateDecoderOnlyOutput(sequences=tensor([[128000,    791,   7438,    315,   2324,    374,    311,   1505,    701,
           7580,    304,   2324,     13,   2057,   1505,    701,  1

In [44]:
import torch
import torch.nn.functional as F
input_text = "I have a wug, now there are two. I have two"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# Get model logits (raw scores)
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits  # Shape: (batch_size, sequence_length, vocab_size)
print("logits shape", logits.shape)
# Extract the logits for the last token
last_token_logits = logits[:, -1, :]  # Shape: (1, vocab_size)

# Convert logits to probabilities using softmax
probs = F.softmax(last_token_logits, dim=-1)  # Shape: (1, vocab_size)

# Get the top 10 most likely tokens and their probabilities
top_k = 10
top_probs, top_indices = torch.topk(probs, top_k)

print("top probs", top_probs.shape)

# Decode tokens and print probabilities
for token_id, prob in zip(top_indices[0], top_probs[0]):
    print(f"{tokenizer.decode([token_id.item()])}: {prob.item():.4f}")

logits shape torch.Size([1, 15, 128256])
top probs torch.Size([1, 10])
 w: 0.3713
 dogs: 0.0305
 cats: 0.0155
 kids: 0.0125
,: 0.0123
 little: 0.0119
 more: 0.0116
 of: 0.0110
 children: 0.0099
 W: 0.0091


In [45]:
print(output_logits)
print(tokenizer.decode(output_logits[0], skip_special_tokens=True))

GenerateDecoderOnlyOutput(sequences=tensor([[128000,    791,   7438,    315,   2324,    374,    311,   1505,    701,
           7580,    304,   2324,     13,   2057,   1505,    701,  11939,     11,
            311,   1505,    701,   8260,     13,   2057,   1505,    701,   9131,
             13,   2057,   1505,    701,   7580,    304,   2324,    627,   3923,
            374,    701,   7580,    304,   2324,     30,   2650,    649,    499,
           1505,    433,     30,   3639,    527]]), scores=(tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]]), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]]), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]]), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]]), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]]), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]]), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]]), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]]), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]]), tensor([[-inf

TypeError: argument 'ids': 'list' object cannot be interpreted as an integer